# Train your own model — the only lever left

You are rank **465 / 4977**, bronze cuts at ~**497**. A **33-place margin**, and the medal
is decided on the **private** board. Rearranging the public models' predictions is
exhausted — `v6a` moved the ordering and the score didn't budge. What's left is adding a
model the public ensemble doesn't contain.

This notebook is **self-contained**: competition dataset only. It mines the labels from
the reports, builds an image cache, trains 5 folds, and tells you honestly whether the
model learned anything.

## Settings — note the first one is different from your other notebooks

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 × 2** |
| Internet | **ON** ← this one only |
| Datasets | just the competition |

Internet is fine here because **this is not your submission notebook.** Only the
submission notebook must be offline. Internet buys pretrained backbone weights, which
matter a great deal with 4407 studies.

## Run it in two passes

1. **`SMOKE_TEST = True`** (the default). ~2 minutes. Processes 40 studies and **plots the
   slices**. I cannot test DICOM reading from here, so look at the pictures: they must
   look like sagittal knee MRI. If they're blank, noise, or the wrong body part, stop and
   send me the plot.
2. **`SMOKE_TEST = False`**, Run All. ~15 min caching + ~80 min training.

## What it does, and why this way

- **One sagittal fluid-sensitive series per study**, 16 slices, 224px, `uint8` memmap
  (~3.5 GB). One series keeps it affordable; sagittal fluid-sensitive is the richest
  single view for menisci, ACL and effusion.
- **Right knees get their slice order reversed**, matching the public pipeline, so
  "medial" and "lateral" mean the same thing on every study.
- **16 slices as 16 input channels.** The first convolution learns to weigh depth. That is
  a genuinely different inductive bias from the ensemble's DINOv2 slot-attention and
  CoAtNet window readers — and *difference* is what makes a blend member worth adding.
- **Soft-target BCE, masked** wherever the report gave no label. The 25 unreadable reports
  are NaN, not twelve fabricated negatives.

## How to read the result

Two numbers are printed. The one that matters is **OOF AUC against the 58 gold studies** —
those were never used in training and are the only unbiased signal available.

- **0.80+** → a useful blend member
- **0.85+** → a strong one
- **below ~0.70** → something is wrong; send me the output rather than blending it

The other number (against the mined labels) is self-consistent, so it only proves the
model fits its own targets. Don't be impressed by it.

Expect the standalone number to be well below 0.943. That's fine and expected — a blend
member needs to be *different and decent*, not better than the whole ensemble.

## After it finishes

**Save Version**, then attach this notebook's **output** to your submission notebook. The
inference + blend step comes next; don't change your two selected submissions until the
new blend actually beats them.


## 0. Config — **start with `SMOKE_TEST = True`**

In [ ]:
# =============================================================================
# CONFIG
# -----------------------------------------------------------------------------
# Self-contained: competition dataset only.  Mines labels from the reports,
# builds a two-plane image cache, trains 5 folds, saves weights.
#
# INTERNET MUST BE ON.  Only the *submission* notebook has to be offline, and
# this is not it.  Internet buys pretrained backbone weights, which matter a
# great deal with 4407 studies.
#
# Accelerator: GPU T4 x2.
# Expected: ~25 min caching, ~60 min training.
# =============================================================================
import os

SMOKE_TEST = True       # 40 studies, ~2 min, verifies the DICOM reader in the LOG.
                        # Then set False and Run All again.

# Two planes, because three of the twelve labels are compartment-specific.
# Medial OA / Lateral OA / PF OA are read off CORONAL images -- a sagittal-only
# model is close to blind on them, and they are a quarter of the metric.
PLANES = (('Sagittal', 12), ('Coronal', 12))
N_CH = sum(k for _, k in PLANES)        # 24 input channels
IMG = 224
FOLDS = 5
EPOCHS = 12
BATCH = 16
LR = 3e-4
BACKBONE = 'convnext_tiny'   # swap to 'tf_efficientnetv2_s' for a 2nd diverse run
TAG = 'sagcor_cnx'           # names the output files; change it per run
SEED = 42

COMP = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
WORK = '/kaggle/working'
# The cache is ~5 GB. /kaggle/temp is NOT committed as notebook output, so it
# must live there -- in /kaggle/working it would be saved on every Save Version.
TMP = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else WORK
CACHE = f'{TMP}/cache_{N_CH}x{IMG}.npy'

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
          'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
          'Contusion', 'Fracture']

print('planes      : %s  -> %d channels' % (list(PLANES), N_CH))
print('backbone    : %s   tag: %s' % (BACKBONE, TAG))
print('cache       : %s' % CACHE)
print('SMOKE_TEST  : %s' % SMOKE_TEST)
if SMOKE_TEST:
    print()
    print('>> smoke mode: 40 studies. Everything prints to the LOG, including an')
    print('>> ASCII rendering of real slices, so "Save & Run All" is enough.')
    print('>> Then set SMOKE_TEST = False and Run All again.')

## 1a. The report miner (same code as the labels notebook, inlined so this
notebook needs nothing but the competition dataset)

In [ ]:
# Report label miner (lexicons + scoring only; main() stripped)
# =============================================================================
# v1 scored 0.6896 mean AUC against the 58 gold studies.  Its diagnostics named
# four concrete causes, all fixed here:
#
#  1. NORMALISATION DESTROYED TEXT.  v1 did re.sub(r'[^a-z0-9...]', ' ', t) after
#     stripping combining marks, which deletes every character that does not
#     decompose to ASCII: Cyrillic, Greek, and Turkish dotless i.  That is why
#     "yirtik" (tear) arrived as "y rt k" and why 31.8% of reports were "unknown".
#     v2 maps the non-decomposing letters explicitly, transliterates Cyrillic,
#     and keeps any letter it cannot map instead of blanking it.
#
#  2. SIX LANGUAGES, NOT TWO.  The misses contained Dutch ("kraakbeenverlies",
#     "botoedeem"), Turkish ("on capraz bag" = ACL, "butunluk kaybi" = loss of
#     integrity), Croatian ("zglobni izljev", "hondromalacija"), French and
#     Portuguese.  All are covered below.
#
#  3. THE TEAR LEXICON WAS TOO PERMISSIVE.  Gold has ACL=0 for "grade 1 injury"
#     and MCL=0 for "suspect grade 2 injury", so low-grade sprains, hedged
#     wording and bare "increased signal" must not count as a tear.
#
#  4. NO COORDINATION.  "rotura de menisco interno y lateral" names both menisci;
#     v1 only matched an adjacent "menisco lateral" and missed it.
#
# Also: an unreadable report now yields NaN, not 0.  Emitting 0 asserts "all
# twelve findings are absent", which is a fabricated negative on every label.
#
# The 58 gold labels are themselves suspect in places -- one report says
# "Moderate joint effusion" with Effusion=0 -- which suggests they were read off
# the images rather than the report.  So there is a ceiling on agreement that no
# pattern work can lift, and 0.80 here is a good result, not a mediocre one.
# =============================================================================
import json
import re
import sys
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
          'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
          'Contusion', 'Fracture']

# Letters that NFD does not decompose to ASCII, so they must be mapped by hand.
CHARMAP = {
    'ı': 'i', 'İ': 'i', 'ĳ': 'ij', 'đ': 'd', 'Đ': 'd', 'ð': 'd', 'Ð': 'd',
    'ł': 'l', 'Ł': 'l', 'ø': 'o', 'Ø': 'o', 'æ': 'ae', 'Æ': 'ae',
    'œ': 'oe', 'Œ': 'oe', 'ß': 'ss', 'þ': 'th', 'Þ': 'th', 'ŋ': 'n',
}
CYRILLIC = {
    'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd', 'е': 'e', 'ё': 'e',
    'ж': 'zh', 'з': 'z', 'и': 'i', 'й': 'i', 'к': 'k', 'л': 'l', 'м': 'm',
    'н': 'n', 'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't', 'у': 'u',
    'ф': 'f', 'х': 'h', 'ц': 'c', 'ч': 'ch', 'ш': 'sh', 'щ': 'sch', 'ъ': '',
    'ы': 'y', 'ь': '', 'э': 'e', 'ю': 'yu', 'я': 'ya', 'і': 'i', 'ї': 'i',
    'є': 'e', 'ґ': 'g',
}
GREEK = {
    'α': 'a', 'β': 'b', 'γ': 'g', 'δ': 'd', 'ε': 'e', 'ζ': 'z', 'η': 'i',
    'θ': 'th', 'ι': 'i', 'κ': 'k', 'λ': 'l', 'μ': 'm', 'ν': 'n', 'ξ': 'x',
    'ο': 'o', 'π': 'p', 'ρ': 'r', 'σ': 's', 'ς': 's', 'τ': 't', 'υ': 'y',
    'φ': 'f', 'χ': 'ch', 'ψ': 'ps', 'ω': 'o',
}


def norm(text):
    """Lowercase, ASCII-ish, but LOSSLESS for letters: nothing is blanked."""
    s = str(text)
    out = []
    for ch in s:
        lo = ch.lower()
        if lo in CHARMAP:
            out.append(CHARMAP[lo])
        elif lo in CYRILLIC:
            out.append(CYRILLIC[lo])
        elif lo in GREEK:
            out.append(GREEK[lo])
        else:
            d = unicodedata.normalize('NFD', lo)
            base = ''.join(c for c in d if unicodedata.category(c) != 'Mn')
            # re-consult the maps: 'i' with tonos decomposes to Greek eta, which
            # still needs transliterating.
            if len(base) == 1 and base in CYRILLIC:
                out.append(CYRILLIC[base])
            elif len(base) == 1 and base in GREEK:
                out.append(GREEK[base])
            elif len(base) == 1 and base in CHARMAP:
                out.append(CHARMAP[base])
            elif base and (base.isascii() or unicodedata.category(base[0]).startswith('L')):
                out.append(base)
            elif unicodedata.category(lo).startswith('L'):
                out.append(lo)          # keep an unmapped letter rather than lose it
            else:
                out.append(' ')
    t = ''.join(out).lower()
    t = re.sub(r'[^\w%.,;:()/\- ]+', ' ', t, flags=re.UNICODE)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()


def readable(t):
    """A report is minable only if normalisation left real words behind."""
    letters = sum(c.isalpha() for c in t)
    return letters >= 60 and len(re.findall(r'[a-z]{4,}', t)) >= 8


def clauses(text):
    parts = re.split(r'(?<=[.;:])\s+|\s*\|\s*|\s*(?:\bpero\b|\bbut\b|\bhowever\b|'
                     r'\bancak\b|\bmaar\b|\bmais\b)\s*', text)
    out = []
    for p in parts:
        p = p.strip(' .;:,-')
        if len(p) > 2:
            out.extend(x.strip() for x in re.split(r'\s+-\s+', p) if len(x.strip()) > 2)
    return out


def alt(*words):
    return '(?:' + '|'.join(words) + ')'


# --------------------------------------------------------------------------- #
# multilingual lexicons: en, es, tr, nl, hr/sr, fr, pt, de, ru
# --------------------------------------------------------------------------- #

NEG = alt(
    r'no\s', r'not\s', r'non\s', r'without\s', r'absent', r'negative\s+for',
    r'free\s+of', r'normal', r'intact', r'preserv', r'unremarkable',
    r'within\s+normal\s+limits', r'\bwnl\b', r'no\s+evidence',
    r'sin\s', r'no\s+hay', r'ausencia\s+de', r'intacto', r'conservad', r'integr',
    r'dentro\s+de\s+limites\s+normales', r'sin\s+signos', r'sin\s+alteraciones',
    r'descarta',
    r'\byok\b', r'normaldir', r'izlenmedi', r'saptanmadi', r'gozlenmedi',
    r'mevcut\s+degil', r'korunmus', r'dogaldir', r'saglam', r'intakttir',
    r'\bgeen\b', r'\bniet\b', r'normaal', r'ongeschonden', r'\bzonder\b', r'\bgaaf\b',
    r'\bbez\b', r'\bnema\b', r'uredan', r'uredna', r'ocuvan', r'\bnije\b', r'intaktan',
    r'pas\s+de', r'\bsans\b', r'absence\s+de', r'respecte', r'conserve',
    r'\bsem\b', r'\bnao\b', r'\bintegro\b', r'preservado',
    r'\bkein', r'unauffallig', r'regelrecht', r'\bohne\b',
    r'\bnet\b', r'sohranen', r'ne\s+opredel', r'bez\s+priznakov',
)

# a definite tear / rupture, in any of the languages
TEAR = alt(
    # tear\w* not tears?\b -- "extensive complete TEARING of the medial
    # meniscus" (gold = 1) was missed by the narrower form.
    r'tear\w*', r'\btorn\b', r'ruptur\w*', r'disrupt\w*', r'discontinuit\w*',
    r'avuls\w*', r'buckethandle', r'bucket-?handle',
    r'rotura', r'\broto\b', r'\brotas?\b', r'desgarro', r'desinsercion',
    r'solucion\s+de\s+continuidad',
    r'yirtik\w*', r'yirtil\w*', r'yirtig\w*', r'butunluk\s+kayb\w*', r'kopma',
    r'scheur\w*', r'gescheurd', r'discontinuiteit',
    r'puknuc\w*', r'raskid', r'diskontinuitet',
    r'dechirure', r'desinsertion',
    r'\briss\b', r'durchtrenn\w*',
    r'razryv', r'povrezhdeni\w*',
)

# wording that must BLOCK a tear: hedges, low grades, plain degeneration.
# Gold: ACL=0 for "grade 1 injury", MCL=0 for "suspect grade 2 injury".
SUPPRESS = alt(
    r'grade\s*[12i]\b', r'grade\s*ii\b', r'grado\s*[12i]\b', r'grade\s*i{1,2}\s',
    r'\bgrade\s*ii\b', r'icrs\s+grade\s*[12i]\b',
    r'suspect\w*', r'suspicious', r'possible', r'possibl\w*', r'question\w*',
    r'cannot\s+(?:be\s+)?exclude', r'sospech\w*', r'posible', r'mogelijk\w*',
    r'olasi\w*', r'kuskulu', r'probable', r'verdenkt', r'suspicion',
    # "degenerat" was here and blocked "complex DEGENERATIVE TEAR of the lateral
    # meniscus", which gold marks 1.  Degeneration is not a TEAR term to begin
    # with, so suppressing on it could only ever lose true positives.
    r'intrasubstance', r'interstitial', r'mucoid', r'\bsprain\b', r'esguince',
    # gold has MCL=0 for "low-grade partial tear of the medial collateral ligament"
    r'low[- ]grade', r'bajo\s+grado', r'laag[- ]gradig',
)
DEFINITE = alt(
    r'complete', r'full[- ]thickness', r'total', r'grade\s*3\b', r'grade\s*iii\b',
    r'grado\s*3\b', r'completa?\b', r'komplet\w*', r'volledig', r'kompletan',
)

NOT_TEAR = alt(
    r'meniscectom\w*', r'amputacion', r'postquirurgic\w*', r'post-?surgical',
    r'postoperator\w*', r'\bresec\w*', r'prior\s+surgery', r'previa\s+cirugia',
    r'meniscectomie', r'rezekci\w*',
)

CARTILAGE = alt(
    r'osteoarthrit\w*', r'osteoartrosis', r'artrosis', r'gonartrosis', r'chondros\w*',
    r'chondropath\w*', r'condropat\w*', r'chondromalac\w*',
    # bare "chondral" matched INSIDE "subchondral", so "a subchondral linear
    # hypointensity over ... medial tibial plateau" fired Medial OA.  Only "sub"
    # is excluded: an OSTEOchondral lesion really is a cartilage finding.
    r'(?<!sub)chondral', r'(?<!sub)condral',
    r'cartilage\s+' + alt('loss', 'defect', 'thinning', 'fissur\\w*', 'ulcer\\w*', 'damage', 'wear'),
    r'(?:perdida|adelgazamiento|defecto|ulcera|fisura)\w*\s+(?:.{0,24})?cartilag\w*',
    r'osteophyt\w*', r'osteofit\w*', r'subchondral\s+' + alt('sclerosis', 'cyst', 'edema', 'oedema'),
    r'esclerosis\s+subcondral', r'degenerative\s+change', r'cambios\s+degenerativos',
    r'joint\s+space\s+narrowing', r'pinzamiento\s+articular',
    # Bare cartilage NOUNS are removed throughout.  "cartilage", "knorpel",
    # "hrskavica", "kikirdak" and "kraakbeen" matched template headers such as
    # "lateral compartment cartilage" and "medial compartment (meniscus,
    # collateral ligament, cartilage)", which is where the three OA labels
    # collected most of their false positives.  A qualifier is now required.
    r'kondromalaz\w*', r'kondropat\w*', r'(?<!sub)kondral',
    r'kikirdak\s+' + alt('kayb\\w*', 'incelme\\w*', 'defekt\\w*', 'hasar\\w*'),
    r'osteoartrit\w*',
    r'kraakbeen' + alt('verlies', 'lijden', 'schade', 'defect'),
    r'chondropathie', r'chondromalacie', r'artrose', r'osteofyt\w*',
    r'hondromalacij\w*', r'gonartro\w*', r'artroz\w*',
    r'(?:denudacij|fisur|erozij|ostecenj|istrosen)\w*\s+\w*\s?hrskavic\w*',
    r'hondropatij\w*', r'goanrtro\w*',
    r'arthrose', r'osteophyte',
    r'cartilage\s+' + alt('use\\w*', 'aminci\\w*', 'fissur\\w*'),
    r'condropatia', r'artrose',
    r'knorpel' + alt('schaden', 'verlust', 'defekt', 'glatze'),
    r'arthros\w*',
    r'hondromalyaci\w*', r'artroz', r'hryasch\w*',
    # bare abbreviations, common in terse reports ("incipient OA of all three
    # compartments" produced nothing at all without this)
    r'\boa\b', r'\bdjd\b',
    r'oa', r'djd',   # bare abbreviations, common in terse reports
)

BOTH_MENISCI = alt(
    r'medial\s+and\s+lateral\s+menisc\w*', r'lateral\s+and\s+medial\s+menisc\w*',
    r'menisco\s+(?:interno|medial)\s+y\s+(?:lateral|externo)',
    r'menisco\s+(?:lateral|externo)\s+y\s+(?:interno|medial)',
    r'meniscos\s+interno\s+y\s+externo', r'ambos\s+meniscos',
    r'medial\s+ve\s+lateral\s+menisk\w*', r'lateral\s+ve\s+medial\s+menisk\w*',
    r'mediale\s+en\s+laterale\s+menisc\w*', r'both\s+menisci',
    r'menisques\s+(?:interne|medial)\s+et\s+(?:externe|lateral)',
)

ENTITY = {
    'ACL': alt(r'\bacl\b', r'\blca\b', r'\bacl[.,]', r'anterior\s+cruciate',
               r'cruzado\s+anterior', r'on\s+capraz\s+bag\w*', r'\bocb\b',
               r'voorste\s+kruisband', r'prednji\s+krizni', r'ligament\s+croise\s+anterieur',
               r'vorderes\s+kreuzband', r'perednyaya\s+krestoobraznaya', r'\bpks\b'),
    'MCL': alt(r'\bmcl\b', r'\blcm\b', r'medial\s+collateral', r'colateral\s+medial',
               r'colateral\s+interno', r'ic\s+yan\s+bag\w*', r'medial\s+kollateral',
               r'mediale\s+collaterale', r'medijalni\s+kolateralni',
               r'ligament\s+collateral\s+medial', r'innenband', r'tibial\s+collateral'),
    'Medial Meniscus': alt(r'medial\s+meniscus', r'menisco\s+medial', r'menisco\s+interno',
                           r'medial\s+menisk\w*', r'ic\s+menisk\w*', r'mediale\s+meniscus',
                           r'medijalni\s+menisk\w*', r'menisque\s+(?:medial|interne)',
                           r'innenmeniskus', r'medialnyi\s+menisk', r'internal\s+meniscus'),
    'Lateral Meniscus': alt(r'lateral\s+meniscus', r'menisco\s+lateral', r'menisco\s+externo',
                            r'lateral\s+menisk\w*', r'dis\s+menisk\w*', r'laterale\s+meniscus',
                            r'lateralni\s+menisk\w*', r'menisque\s+(?:lateral|externe)',
                            r'aussenmeniskus', r'lateralnyi\s+menisk', r'external\s+meniscus'),
}

PF_CTX = alt(r'patellofemoral', r'patelofemoral', r'femoropatelar', r'femoro-?patelar',
             r'patell\w*', r'rotulian\w*', r'rotula', r'trochlea\w*', r'troclea\w*',
             r'patela\w*', r'trohlej\w*', r'patellofemoraln\w*', r'kniescheibe',
             r'retropatell\w*', r'patellaire')
MED_CTX = alt(r'medial\s+' + alt('compartment', 'femorotibial', 'tibiofemoral',
                                 'femoral\\s+condyle', 'tibial\\s+plateau', 'joint'),
              r'compartimento\s+(?:femorotibial\s+)?medial', r'femorotibial\s+medial',
              r'condilo\s+femoral\s+medial', r'vertiente\s+medial',
              r'medial\s+kompartman', r'mediale\s+compartiment',
              r'medijalnog?\s+kompartment\w*', r'compartiment\s+medial',
              r'mediaal\s+femorotibiaal', r'mediale\s+femorotibiaal')
LAT_CTX = alt(r'lateral\s+' + alt('compartment', 'femorotibial', 'tibiofemoral',
                                  'femoral\\s+condyle', 'tibial\\s+plateau', 'joint'),
              r'compartimento\s+(?:femorotibial\s+)?lateral', r'femorotibial\s+lateral',
              r'condilo\s+femoral\s+lateral', r'lateral\s+kompartman',
              r'laterale\s+compartiment', r'lateralnog?\s+kompartment\w*',
              r'compartiment\s+lateral', r'lateraal\s+femorotibiaal',
              r'laterale\s+femorotibiaal', r'laterale\s+tibiaplateau')
ALL_COMPART = alt(r'tricompartimental', r'tricompartmental', r'gonartrosis', r'gonartro\w*',
                  r'global\s+osteoarthrit\w*', r'all\s+three\s+compartmen\w*',
                  r'three\s+compartmen\w*', r'todos\s+los\s+compartiment\w*',
                  r'tum\s+kompartman\w*', r'alle\s+drie')

PRESENCE = {
    'Effusion': alt(r'joint\s+effusion', r'effusion', r'hydrarthros\w*', r'hidrartros\w*',
                    r'fluid\s+in\s+the\s+joint', r'intra-?articular\s+fluid',
                    r'derrame\s+articular', r'\bderrame\b', r'liquido\s+articular',
                    r'eklem\s+ici\s+sivi', r'eklem\s+sivi\w*', r'efuzyon', r'hidrartroz',
                    r'sivi\s+miktar\w*', r'gewrichtsvocht', r'hydrops', r'effusie',
                    r'vochtcollectie', r'zglobni\s+izljev', r'\bizljev\w*',
                    r'epanchement', r'gelenkerguss', r'\berguss\b', r'\bvypot\b',
                    r'suprapatellar\s+(?:bursa\s+)?distension'),
    'Synovitis': alt(r'synovit\w*', r'sinovit\w*', r'synovial\s+' + alt('thickening', 'thickend',
                     'thickened', 'proliferation', 'hypertroph\\w*', 'enhancement'),
                     r'engrosamiento\s+sinovial', r'hipertrofia\s+sinovial', r'pannus',
                     r'bursit\w*', r'burzit\w*', r'plica\s+syndrome',
                     r'sinovyal\s+kalinlas\w*', r'synoviale\s+verdikking',
                     r'proliferacij\w*\s+sinovij\w*', r'sinovijaln\w*',
                     r'synovite', r'epaississement\s+synovial', r'synovialitis',
                     r'sinovialn\w*\s+prolifer\w*'),
    "Baker's": alt(r'baker\w*', r'popliteal\s+cyst', r'quiste\s+popliteo',
                   r'quistes\s+popliteos', r'popliteal\s+fossa\s+cyst',
                   r'gastrocnemio-?semimembranos\w*', r'poplital\s+kist',
                   r'popliteal\s+kist', r'popliteale\s+cyste', r'poplitealn\w*\s+cist\w*',
                   r'bakerov\w*\s+cist\w*', r'kyste\s+poplite', r'cisto\s+popliteo',
                   r'poplitealzyste', r'kista\s+beikera'),
    # "osteochondral fracture" is excluded on purpose: gold report 2 names one and
    # still has Fracture=0, so the graders count it as a chondral injury.
    'Fracture': alt(r'(?<!osteochondral\s)(?<!osteocondral\s)(?<!osteokondral\s)fractur\w*',
                    r'fisura\s+osea', r'avulsion\s+fragment', r'\bkirik\w*', r'fraktur\w*',
                    r'impactiefractuur', r'\bbreuk\b', r'prijelom', r'impresijsk\w*\s+fraktur\w*',
                    r'\bfratura\b', r'\bbruch\b', r'perelom'),
    # NB: "subchondral bone edema" is deliberately absent.  Next to a cartilage
    # defect it is a degenerative sign, not a traumatic contusion -- gold report 2
    # carries three of them and has Contusion=0.
    'Contusion': alt(r'contusion\w*', r'kontuzij\w*', r'kontuzyon\w*', r'contusao',
                     r'bone\s+' + alt('bruise', 'bruising'),
                     r'(?<!subchondral\s)(?<!subkondral\s)(?:bone\s+)?marrow\s+' + alt('edema', 'oedema'),
                     r'edema\s+(?:de\s+)?(?:la\s+)?medula\s+osea', r'edema\s+oseo',
                     r'kemik\s+ilig\w*\s+odem\w*', r'kemik\s+odem\w*',
                     r'botcontusie', r'botoedeem', r'beenmergoedeem',
                     r'kostani\s+edem', r'oedeme\s+osseux', r'knochenmarkodem',
                     r'knochenprellung', r'otek\s+kostnogo'),
}


# Severity gate -- a hypothesis test, not a fix.
# The surviving errors are overwhelmingly cases where the report states a finding
# plainly and gold says absent: "moderate joint effusion", "mild synovitis",
# "trace baker s cyst", "low-grade cartilage fissuring".  That pattern suggests
# gold applies a severity threshold a bare mention does not meet.  Rather than
# assume it, main() scores WITH and WITHOUT this gate and reports both.
MILD = alt(
    r'\bmild\w*', r'\bsmall\b', r'\btrace\b', r'\bminimal\w*', r'\bslight\w*',
    r'\btiny\b', r'\bscant\w*', r'low[- ]grade', r'\bmicro',
    r'\bleve\b', r'discret\w*', r'pequen\w*', r'\bligero\w*',
    r'\bhafif\w*', r'\bgering\w*', r'\bleicht\w*',
    r'\bblago\b', r'\bminimaln\w*', r'\bmanji\b', r'\bfaible\w*', r'\bleger\w*',
    r'\bnebol\w*',
)


def _near(clause, span, pattern, window=110):
    lo = max(0, span[0] - window)
    hi = min(len(clause), span[1] + window)
    return re.search(pattern, clause[lo:hi]) is not None


# Trailing negation must be far stricter than leading negation.  With the full
# NEG list over a 65-character window, "subchondral insufficiency fracture at the
# medial tibial plateau WITHOUT articular surface collapse" read as a negated
# fracture (gold = 1): the "without" negates the collapse, not the fracture.
# Only "this structure is normal" phrasings may negate from the right, and only
# close by.
NEG_POST = alt(
    r'normal', r'intact', r'preserv', r'unremarkable', r'conservad', r'integr',
    r'intacto', r'normaldir', r'korunmus', r'dogaldir', r'saglam',
    r'normaal', r'ongeschonden', r'uredan', r'uredna', r'ocuvan', r'intaktan',
    r'conserve', r'respecte', r'preservado', r'integro', r'unauffallig',
    r'regelrecht', r'sohranen', r'izlenmedi', r'saptanmadi', r'\byok\b',
    r'\bgeen\b', r'sin\s+alteraciones', r'sin\s+signos', r'sin\s+rotura',
    r'within\s+normal\s+limits',
)


def negated(clause, span):
    before = clause[max(0, span[0] - 95):span[0]]
    after = clause[span[1]:span[1] + 32]
    if re.search(NEG + r'[^.;:]{0,65}$', before):
        return True
    if re.search(r'^[^.;:]{0,32}\b' + NEG_POST, after):
        return True
    return False


def hits(clause, pattern):
    return [m.span() for m in re.finditer(pattern, clause)]


def score_report(text, want_evidence=False, severity_gate=False):
    """label -> support count.  NaN-able callers check readable() first."""
    t = norm(text)
    score = {l: 0 for l in LABELS}
    ev = {l: [] for l in LABELS}

    def add(label, clause, why):
        score[label] += 1
        if want_evidence and len(ev[label]) < 3:
            ev[label].append('%s  <<%s>>' % (clause[:170], why))

    for c in clauses(t):
        if severity_gate and re.search(MILD, c) and not re.search(DEFINITE, c):
            continue        # a hedged-severity clause contributes nothing
        pf = bool(re.search(PF_CTX, c))
        tear_spans = [s for s in hits(c, TEAR) if not negated(c, s)]
        definite = bool(re.search(DEFINITE, c))

        # entity + abnormality
        for label, ent in ENTITY.items():
            for span in hits(c, ent):
                if negated(c, span):
                    continue
                if not tear_spans:
                    continue
                if 'Meniscus' in label and re.search(NOT_TEAR, c):
                    continue
                # a hedge or a low grade blocks it unless the clause also says
                # complete / full-thickness / grade 3
                if not definite and any(_near(c, ts, SUPPRESS) for ts in tear_spans):
                    continue
                add(label, c, label + ' tear')
                break

        # "medial and lateral meniscus" names both at once
        for span in hits(c, BOTH_MENISCI):
            if negated(c, span) or not tear_spans or re.search(NOT_TEAR, c):
                continue
            if not definite and any(_near(c, ts, SUPPRESS) for ts in tear_spans):
                continue
            add('Medial Meniscus', c, 'both menisci')
            add('Lateral Meniscus', c, 'both menisci')
            break

        # findings that are their own entity
        for label, pat in PRESENCE.items():
            for span in hits(c, pat):
                if negated(c, span):
                    continue
                add(label, c, label + ' present')
                break

        # cartilage, routed by compartment; patellofemoral wording wins, so
        # "lateral patellar facet" is PF OA and not Lateral OA
        cart = [s for s in hits(c, CARTILAGE) if not negated(c, s)]
        if cart:
            if pf:
                add('PF OA', c, 'PF cartilage')
            if re.search(MED_CTX, c) and not pf:
                add('Medial OA', c, 'medial cartilage')
            if re.search(LAT_CTX, c) and not pf:
                add('Lateral OA', c, 'lateral cartilage')
            if re.search(ALL_COMPART, c):
                for l in ('Medial OA', 'Lateral OA', 'PF OA'):
                    add(l, c, 'all compartments')
    return (score, ev) if want_evidence else score


# --------------------------------------------------------------------------- #
# language guess
# --------------------------------------------------------------------------- #

LANG_CUES = {
    'english': r'\b(?:the|and|with|there|is|are|meniscus|knee|findings|signal|joint)\b',
    'spanish': r'\b(?:del|los|sin|articular|menisco|rodilla|hallazgos|senal|con)\b',
    'turkish': r'\b(?:\bve\b|yok|mevcut|bulgular|menisk\w*|izlenmistir|diz|bag)\b',
    'dutch': r'\b(?:van|niet|geen|bevindingen|knie|kraakbeen|met|rechts|links)\b',
    'croatian': r'\b(?:\bje\b|\bse\b|\bna\b|zgloba|hrskavic\w*|menisk\w*|promjene|bez)\b',
    'french': r'\b(?:genou|sans|avec|les|des|menisque|epanchement)\b',
    'portuguese': r'\b(?:\bnao\b|\bdos\b|joelho|achados|sem|menisco\b)\b',
    'german': r'\b(?:und|nicht|kein\w*|kniegelenk|knorpel|meniskus)\b',
    'russian': r'\b(?:\bi\b|\bne\b|sustav\w*|menisk\w*|razryv|kolennogo)\b',
}


def guess_lang(t):
    counts = {k: len(re.findall(v, t)) for k, v in LANG_CUES.items()}
    best = max(counts, key=counts.get)
    return best if counts[best] >= 3 else 'unknown'


# --------------------------------------------------------------------------- #
# evaluation
# --------------------------------------------------------------------------- #

def auc(y, s):
    y, s = np.asarray(y, float), np.asarray(s, float)
    ok = np.isfinite(y) & np.isfinite(s)
    y, s = y[ok], s[ok]
    p, n = y == 1, y == 0
    if p.sum() == 0 or n.sum() == 0:
        return None
    order = np.argsort(s, kind='stable')
    r = np.empty(len(s), float)
    r[order] = np.arange(1, len(s) + 1)
    u, inv, cnt = np.unique(s, return_inverse=True, return_counts=True)
    if len(u) != len(s):
        acc = np.zeros(len(u))
        np.add.at(acc, inv, r)
        r = (acc / cnt)[inv]
    return float((r[p].sum() - p.sum() * (p.sum() + 1) / 2) / (p.sum() * n.sum()))


## 1b. Build the soft label matrix

In [ ]:
# =============================================================================
# 1b. SOFT LABELS from the mined reports
# -----------------------------------------------------------------------------
# train.csv carries real labels for 58 of 4407 studies; the rest have a
# free-text report.  The miner agrees with those 58 at ~0.75 AUC, which is
# noisy -- but a training target does not need the right THRESHOLD, only the
# right ORDERING, and ordering is what AUC measures.  Unreadable reports become
# NaN and are masked out of the loss rather than asserted as twelve negatives.
# =============================================================================
import numpy as np
import pandas as pd

train = pd.read_csv(f'{COMP}/train.csv', dtype={'StudyInstanceUID': str})
texts = [norm(t) for t in train['Report'].astype(str)]
ok = np.array([readable(t) for t in texts])
rows = [score_report(t) if o else {l: np.nan for l in LABELS} for t, o in zip(texts, ok)]
Y = np.clip(pd.DataFrame(rows)[LABELS].to_numpy(float) / 2.0, 0, 1)
uids_all = train['StudyInstanceUID'].astype(str).tolist()
gold = train.dropna(subset=LABELS).set_index('StudyInstanceUID')[LABELS]
print('labels: %d studies, %d minable, %d gold' % (len(Y), int(ok.sum()), len(gold)))
for j, l in enumerate(LABELS):
    print('   %-18s positive rate %.4f' % (l, float(np.nanmean(Y[:, j] > 0))))

## 2. Image cache

One sagittal stack per study → a `uint8` memmap. In smoke mode this stops after 40 studies and shows you the slices.

In [ ]:
# =============================================================================
# 2. TWO-PLANE IMAGE CACHE -> uint8 memmap
# -----------------------------------------------------------------------------
# Per study: 12 sagittal slices then 12 coronal slices, stacked as 24 channels.
#
# Laterality is normalised so "medial" and "lateral" mean the same thing on
# every study, matching what the public pipeline does:
#   sagittal -> reverse the slice order for a right knee
#   coronal  -> mirror left-right for a right knee
# Without this the two OA compartment labels are unlearnable.
# =============================================================================
import glob
import time
import pydicom
import cv2
from concurrent.futures import ThreadPoolExecutor

ser = pd.read_csv(f'{COMP}/train_series.csv',
                  dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})


def pick(df, plane):
    """Fluid-sensitive series of this plane; else any of this plane; else None."""
    for q in (df[(df.Anatomical_Plane == plane) & (df.Fluid_Sensitive == 1)],
              df[df.Anatomical_Plane == plane]):
        if len(q):
            return q.iloc[0].SeriesInstanceUID
    return None


chosen, _miss = {}, {p: 0 for p, _ in PLANES}
for u, g in ser.groupby('StudyInstanceUID'):
    picks = {}
    for plane, _ in PLANES:
        s = pick(g, plane)
        if s is None:
            _miss[plane] += 1
        picks[plane] = s
    chosen[u] = picks
print('studies: %d' % len(chosen))
for p, n in _miss.items():
    print('  missing %-9s %d  (falls back to the other plane)' % (p, n))


def order_files(d):
    """Sort by physical position along the slice normal; filename as fallback."""
    out = []
    for f in glob.glob(f'{d}/*.dcm'):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True, force=True)
            ipp = [float(x) for x in getattr(h, 'ImagePositionPatient', [0, 0, 0])]
            iop = [float(x) for x in getattr(h, 'ImageOrientationPatient',
                                             [1, 0, 0, 0, 1, 0])]
            lat = str(getattr(h, 'Laterality', '') or
                      getattr(h, 'ImageLaterality', '') or '')
            out.append((float(np.dot(ipp, np.cross(iop[:3], iop[3:]))), f, lat))
        except Exception:
            out.append((0.0, f, ''))
    if not out:
        return [], ''
    if len({o[0] for o in out}) < max(2, len(out) // 2):
        out.sort(key=lambda x: x[1])
    else:
        out.sort(key=lambda x: x[0])
    return [o[1] for o in out], next((o[2] for o in out if o[2]), '')


def read_stack(root, uid, sid, k, plane):
    """-> (k, IMG, IMG) float32 in [0,1], or None."""
    files, lat = order_files(f'{root}/{uid}/{sid}')
    if len(files) < 4:
        return None
    right = lat.upper().startswith('R')
    if right and plane == 'Sagittal':
        files = files[::-1]
    lo, hi = int(0.15 * len(files)), int(0.85 * len(files))
    idx = np.linspace(lo, max(lo + 1, hi - 1), k).astype(int)
    out = np.zeros((k, IMG, IMG), np.float32)
    for n, i in enumerate(idx):
        try:
            a = pydicom.dcmread(files[i], force=True).pixel_array.astype(np.float32)
        except Exception:
            continue
        p1, p99 = np.percentile(a, [1, 99])
        a = np.clip((a - p1) / max(1e-6, p99 - p1), 0, 1)
        h, w = a.shape
        s = min(h, w)
        a = a[(h - s) // 2:(h - s) // 2 + s, (w - s) // 2:(w - s) // 2 + s]
        a = cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)
        if right and plane != 'Sagittal':
            a = a[:, ::-1]
        out[n] = a
    return out


def load_study(uid, root):
    """-> (N_CH, IMG, IMG) uint8, or None if no plane could be read."""
    picks = chosen.get(uid) or {}
    got, fallback = {}, None
    for plane, k in PLANES:
        sid = picks.get(plane)
        st = read_stack(root, uid, sid, k, plane) if sid else None
        got[plane] = st
        if st is not None and fallback is None:
            fallback = (plane, st)
    if fallback is None:
        return None
    parts = []
    for plane, k in PLANES:
        st = got[plane]
        if st is None:                      # reuse whichever plane we did get,
            src = fallback[1]               # resampled to this plane's budget
            sel = np.linspace(0, len(src) - 1, k).astype(int)
            st = src[sel]
        parts.append(st)
    return (np.concatenate(parts, 0) * 255).astype(np.uint8)


work_uids = uids_all[:40] if SMOKE_TEST else uids_all
cache = np.lib.format.open_memmap(CACHE, mode='w+', dtype=np.uint8,
                                  shape=(len(work_uids), N_CH, IMG, IMG))
have = np.zeros(len(work_uids), bool)
t0 = time.time()


def fill(i):
    v = load_study(work_uids[i], f'{COMP}/train_series')
    if v is not None:
        cache[i] = v
        have[i] = True


with ThreadPoolExecutor(max_workers=8) as ex:
    for n, _ in enumerate(ex.map(fill, range(len(work_uids)))):
        if n and n % 250 == 0:
            el = time.time() - t0
            print('  %5d/%d  %.0fs elapsed, ~%.0fs left'
                  % (n, len(work_uids), el, el / n * (len(work_uids) - n)), flush=True)
cache.flush()
print('cached %d/%d studies in %.0fs'
      % (int(have.sum()), len(work_uids), time.time() - t0))

if SMOKE_TEST:
    # Everything below prints to the LOG on purpose: "Save & Run All" shows the
    # log but hides plots, and the whole point is to verify the DICOM reader
    # before committing to a long run.
    def ascii_thumb(img, w=56, h=24):
        a = cv2.resize(img.astype(np.float32), (w, h), interpolation=cv2.INTER_AREA)
        lo, hi = float(a.min()), float(a.max())
        a = (a - lo) / max(1e-6, hi - lo)
        ramp = ' .:-=+*#%@'
        rows = ['   |' + ''.join(ramp[int(v * (len(ramp) - 1))] for v in row) + '|'
                for row in a]
        return chr(10).join(rows)

    _k = np.where(have)[0]
    _v = cache[_k].astype(np.float32)
    print()
    print('=' * 70)
    print('NUMERIC SANITY CHECKS')
    print('=' * 70)
    print('  dark background pixels   %.3f   (expect 0.30-0.75: knee in air)'
          % float((_v < 25).mean()))
    print('  mid-tone tissue pixels   %.3f   (expect 0.25-0.65)'
          % float(((_v >= 25) & (_v < 250)).mean()))
    print('  saturated pixels         %.3f   (expect < 0.10)'
          % float((_v >= 250).mean()))
    print('  flat (blank) studies     %d/%d   (must be 0)'
          % (int(sum(cache[i].std() < 3 for i in _k)), len(_k)))
    _sag = slice(0, PLANES[0][1])
    _cor = slice(PLANES[0][1], N_CH)
    _adj = [float(np.corrcoef(cache[i, s].ravel(), cache[i, s + 1].ravel())[0, 1])
            for i in _k[:12] for s in range(PLANES[0][1] - 1)]
    _end = [float(np.corrcoef(cache[i, 0].ravel(),
                              cache[i, PLANES[0][1] - 1].ravel())[0, 1]) for i in _k]
    print('  adjacent-slice corr      %.3f   (expect 0.5-0.98: similar, not identical)'
          % float(np.nanmean(_adj)))
    print('  first-vs-last sagittal   %.3f   (must be clearly LOWER than adjacent,'
          % float(np.nanmean(_end)))
    print('                                    else the stack is not traversing depth)')
    _cross = [float(np.corrcoef(cache[i, PLANES[0][1] // 2].ravel(),
                                cache[i, PLANES[0][1] + PLANES[1][1] // 2].ravel())[0, 1])
              for i in _k]
    print('  sagittal-vs-coronal corr %.3f   (should be LOW ~0.0-0.6: genuinely'
          % float(np.nanmean(_cross)))
    print('                                    different views, not a duplicate)')
    print()
    print('=' * 70)
    print('ASCII RENDERING -- middle sagittal and middle coronal slice')
    print('Expect a bright blob filling much of the frame on a dark background.')
    print('=' * 70)
    for i in _k[:2]:
        print('  study %s   SAGITTAL' % work_uids[i][:26])
        print(ascii_thumb(cache[i, PLANES[0][1] // 2]))
        print('  study %s   CORONAL' % work_uids[i][:26])
        print(ascii_thumb(cache[i, PLANES[0][1] + PLANES[1][1] // 2]))
        print()
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(2, 6, figsize=(14, 5))
        for c in range(6):
            ax[0, c].imshow(cache[_k[0], c * 2], cmap='gray'); ax[0, c].axis('off')
            ax[1, c].imshow(cache[_k[0], PLANES[0][1] + c * 2], cmap='gray')
            ax[1, c].axis('off')
        plt.suptitle('top: sagittal   bottom: coronal'); plt.tight_layout(); plt.show()
    except Exception:
        pass
    print('=' * 70)
    print('SMOKE TEST DONE. If the renderings look like knee MRI:')
    print('  set SMOKE_TEST = False (top cell) and Run All again.')
    print('The cells below skip themselves while SMOKE_TEST is True.')
    print('=' * 70)

## 3. Train 5 folds

In [ ]:
# =============================================================================
# 3. TRAIN 5 FOLDS
# -----------------------------------------------------------------------------
# The 24 slices enter as 24 input channels, so the first convolution learns to
# weigh depth and plane.  That is a different inductive bias from the public
# ensemble's DINOv2 slot-attention and CoAtNet window readers, and difference is
# what makes a blend member worth adding.
#
# Soft-target BCE, masked wherever the report gave no label.
# =============================================================================
if SMOKE_TEST:
    print('[smoke] training skipped -- set SMOKE_TEST = False to run it')
else:
    import timm
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader

    torch.manual_seed(SEED); np.random.seed(SEED)
    dev = 'cuda'
    Yw = Y[:len(work_uids)]
    usable = np.where(have & np.isfinite(Yw).any(axis=1))[0]
    print('training on %d studies, %d channels' % (len(usable), N_CH))

    class DS(Dataset):
        def __init__(self, idx, aug):
            self.idx, self.aug = idx, aug

        def __len__(self):
            return len(self.idx)

        def __getitem__(self, k):
            i = self.idx[k]
            x = cache[i].astype(np.float32) / 255.0
            if self.aug:
                if np.random.rand() < 0.5:
                    x = x[:, :, ::-1].copy()
                sh = np.random.randint(-2, 3)
                if sh:
                    # Shift depth WITHOUT wrapping. np.roll would move the last
                    # slices to the front, teaching the net that knee depth is
                    # circular, and it would cross the sagittal/coronal boundary.
                    a, b = PLANES[0][1], N_CH
                    def shift(blk, s):
                        if s > 0:
                            return np.concatenate([np.repeat(blk[:1], s, 0), blk[:-s]], 0)
                        return np.concatenate([blk[-s:], np.repeat(blk[-1:], -s, 0)], 0)
                    x = np.concatenate([shift(x[:a], sh), shift(x[a:b], sh)], 0)
                x = x * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.05, 0.05)
            # Per-sample standardisation: MRI intensity has no absolute meaning
            # across scanners or sequences, so raw 0-1 values would waste the
            # pretrained backbone's early layers absorbing that offset.
            x = (x - x.mean()) / (x.std() + 1e-6)
            y = Yw[i].astype(np.float32)
            m = np.isfinite(y).astype(np.float32)
            return (torch.from_numpy(np.ascontiguousarray(x)),
                    torch.from_numpy(np.nan_to_num(y)), torch.from_numpy(m))

    def auc(y, s):
        y, s = np.asarray(y, float), np.asarray(s, float)
        k = np.isfinite(y) & np.isfinite(s)
        y, s = y[k], s[k]
        p, n = y == 1, y == 0
        if p.sum() == 0 or n.sum() == 0:
            return None
        o = np.argsort(s, kind='stable')
        r = np.empty(len(s), float); r[o] = np.arange(1, len(s) + 1)
        u, inv, c = np.unique(s, return_inverse=True, return_counts=True)
        if len(u) != len(s):
            a = np.zeros(len(u)); np.add.at(a, inv, r); r = (a / c)[inv]
        return float((r[p].sum() - p.sum() * (p.sum() + 1) / 2) / (p.sum() * n.sum()))

    folds = np.random.default_rng(SEED).permutation(len(usable)) % FOLDS
    oof = np.full((len(work_uids), len(LABELS)), np.nan)

    for f in range(FOLDS):
        tr, va = usable[folds != f], usable[folds == f]
        model = timm.create_model(BACKBONE, pretrained=True,
                                  in_chans=N_CH, num_classes=len(LABELS)).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        dl = DataLoader(DS(tr, True), batch_size=BATCH, shuffle=True,
                        num_workers=2, drop_last=True, pin_memory=True)
        vl = DataLoader(DS(va, False), batch_size=BATCH * 2, num_workers=2)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, EPOCHS * len(dl))
        scaler = torch.cuda.amp.GradScaler()
        for ep in range(EPOCHS):
            model.train(); tot = 0.0
            for x, y, m in dl:
                x, y, m = x.to(dev), y.to(dev), m.to(dev)
                opt.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast():
                    loss = (F.binary_cross_entropy_with_logits(
                        model(x), y, reduction='none') * m).sum() / m.sum().clamp(min=1)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                sched.step(); tot += float(loss)
            print('  fold %d epoch %2d  loss %.4f' % (f, ep, tot / len(dl)), flush=True)
        model.eval(); P = []
        with torch.no_grad(), torch.cuda.amp.autocast():
            for x, _, _ in vl:
                x = x.to(dev)
                # hflip TTA, same as will be used at inference
                p = torch.sigmoid(model(x)) + torch.sigmoid(model(torch.flip(x, [3])))
                P.append((p / 2).float().cpu().numpy())
        oof[va] = np.concatenate(P)
        torch.save(model.state_dict(), f'{WORK}/{TAG}_f{f}.pt')
        del model; torch.cuda.empty_cache()
        print('fold %d done' % f, flush=True)

## 4. Evaluate — the number that matters is the one against the 58 gold studies

In [ ]:
# =============================================================================
# 4. DID IT LEARN ANYTHING?
# -----------------------------------------------------------------------------
# Two yardsticks.  The second is the one that counts: the 58 GOLD studies were
# never used in training, so they are the only unbiased signal available.
# =============================================================================
if SMOKE_TEST:
    print('[smoke] evaluation skipped -- set SMOKE_TEST = False to run it')
else:
    import json
    print('OOF AUC vs the mined labels (self-consistent -- a fitting check only)')
    v = []
    for j, l in enumerate(LABELS):
        a = auc((Yw[:, j] > 0).astype(float), oof[:, j])
        if a: v.append(a)
        print('   %-18s %s' % (l, 'n/a' if a is None else '%.4f' % a))
    print('   MEAN %.4f' % float(np.mean(v)))

    gi = [i for i, u in enumerate(work_uids)
          if u in gold.index and np.isfinite(oof[i]).all()]
    print('\nOOF AUC vs the %d GOLD studies   <-- THE REAL NUMBER' % len(gi))
    gv, per = [], {}
    for j, l in enumerate(LABELS):
        a = auc(pd.to_numeric(gold.loc[[work_uids[i] for i in gi], l],
                              errors='coerce').to_numpy(float), oof[gi, j])
        per[l] = a
        if a: gv.append(a)
        print('   %-18s %s' % (l, 'n/a' if a is None else '%.4f' % a))
    mean_gold = float(np.mean(gv)) if gv else float('nan')
    print('   MEAN %.4f' % mean_gold)
    print('\n  0.80+ -> a useful blend member.  0.85+ -> a strong one.')
    print('  Below ~0.70 -> something is wrong; send me the output, do not blend it.')
    print('  It does NOT need to beat 0.943 alone -- it needs to be DIFFERENT and decent.')

    np.save(f'{WORK}/{TAG}_oof.npy', oof)
    pd.DataFrame({'StudyInstanceUID': work_uids}).to_csv(
        f'{WORK}/{TAG}_uids.csv', index=False)
    json.dump({'tag': TAG, 'backbone': BACKBONE, 'planes': [list(p) for p in PLANES],
               'n_ch': N_CH, 'img': IMG, 'folds': FOLDS, 'epochs': EPOCHS,
               'gold_auc': mean_gold, 'gold_auc_per_label': per, 'labels': LABELS},
              open(f'{WORK}/{TAG}_card.json', 'w'), indent=2)
    print('\nsaved %s_f0..%d.pt, %s_card.json' % (TAG, FOLDS - 1, TAG))
    print('NEXT: Save Version, then attach THIS notebook\'s OUTPUT to your')
    print('submission notebook and paste in the inference cell.')